***

# **Health_4 Rewrite**

***

Rewriting some of the code Seth created for Health_4 into Python from R.

***

## **Packages**

***

In [3]:
import pandas as pd
import os
import numpy as np
from functools import partial
import re
import glob


***

## **Translating**

***

In [24]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_data = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators','Seth', 'HealthMetrics', 'CountyLifeExp')
path_git = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')
path_health  = os.path.join(path_git, 'Python Code', 'Health')

<>:5: SyntaxWarning: invalid escape sequence '\R'
<>:5: SyntaxWarning: invalid escape sequence '\R'
C:\Users\jchoy\AppData\Local\Temp\ipykernel_31472\2941057322.py:5: SyntaxWarning: invalid escape sequence '\R'
  path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')


In [5]:
# Defining sacog list for subsetting

sacog = ["El Dorado", "Placer", "Sacramento", "Sutter", "Yolo", "Yuba"]
sacog = [f"{county} County (California)" for county in sacog]

In [71]:
# Should be noted, the entire CountyLifeExp folder from Seth's file in Process Revamp > Task 8 needs to be downloaded for this to work. 
folder_path = r'C:\Users\jchoy\Documents\CountyLifeExp'

os.chdir(folder_path)

# Glob just finds all the csvs in a folder 

files = glob.glob('*.csv')

lexp = pd.concat([pd.read_csv(file) for file in files]).reset_index()

# Selecting the needed cols
lexp = lexp[['year', 'location_name', 'race_name', 'age_name', 'val', 'upper', 'lower']]
lexp['location_name'] = lexp['location_name'].str.replace(" County \\(California\\)", "")
lexp.columns = ["Year", "County", "Race", "Age", "Estimate", "Upper", "Lower"]

# Just making it into a pd df and subsetting
lexp = pd.DataFrame(lexp)
lexp = lexp[lexp['County'].isin(sacog)]

print(lexp.head())

   Year                    County    Race      Age   Estimate      Upper  \
0  2000  United States of America   Total  <1 year  76.780325  76.806569   
1  2000  United States of America  Latino  <1 year  79.521017  79.796012   
2  2000  United States of America   Black  <1 year  71.441338  71.541739   
3  2000  United States of America   White  <1 year  77.268962  77.307702   
4  2000  United States of America    AIAN  <1 year  73.122530  74.721068   

       Lower  
0  76.751931  
1  79.263588  
2  71.340015  
3  77.229175  
4  71.617694  


In [79]:
# Calculating mean for specific cols and creating the total dfs on county, race and year
numeric_columns = ['Estimate', 'Upper', 'Lower']
cty_total = lexp[lexp['Age'] == "<1 year"].groupby(['Year', 'County', 'Race'])[numeric_columns].mean().reset_index()
cty_total['Age'] = "All Ages Total"
cty_total = cty_total[['Year', 'County', 'Race', 'Age'] + numeric_columns]  # Reorder columns

sacog_total = lexp[lexp['Age'] == "<1 year"].groupby(['Year', 'Race'])[numeric_columns].mean().reset_index()
sacog_total['County'] = "SACOG Total"
sacog_total['Age'] = "All Ages Total"
sacog_total = sacog_total[['Year', 'County', 'Race', 'Age'] + numeric_columns]  # Reorder columns

In [82]:
# Displaying

display(lexp.head(10))

display(cty_total.head(10))

display(sacog_total.head(10))

,Year,County,Race,Age,Estimate,Upper,Lower
22962,2000,El Dorado County (California),Total,<1 year,78.783667,79.171027,78.414117
22963,2000,El Dorado County (California),Latino,<1 year,82.550839,84.104154,81.132208
22964,2000,El Dorado County (California),Black,<1 year,79.981221,82.328612,77.911071
22965,2000,El Dorado County (California),White,<1 year,78.507125,78.904639,78.121014
22966,2000,El Dorado County (California),AIAN,<1 year,80.163388,84.236030,76.906655
22967,2000,El Dorado County (California),API,<1 year,83.199340,84.601953,81.857021
23094,2000,Placer County (California),Total,<1 year,79.078167,79.374723,78.801713
23095,2000,Placer County (California),Latino,<1 year,80.187144,81.166700,79.323371
23096,2000,Placer County (California),Black,<1 year,77.650525,79.139333,76.251684
23097,2000,Placer County (California),White,<1 year,78.924825,79.218662,78.646113


,Year,County,Race,Age,Estimate,Upper,Lower
0,2000,El Dorado County (California),AIAN,All Ages Total,80.431622,84.798511,76.929894
1,2000,El Dorado County (California),API,All Ages Total,83.191716,84.941544,81.546109
2,2000,El Dorado County (California),Black,All Ages Total,80.152209,83.037950,77.694494
3,2000,El Dorado County (California),Latino,All Ages Total,82.761291,84.776567,81.081637
4,2000,El Dorado County (California),Total,All Ages Total,78.792816,79.268382,78.312337
5,2000,El Dorado County (California),White,All Ages Total,78.519252,78.999080,78.026312
6,2000,Placer County (California),AIAN,All Ages Total,79.401997,83.717448,76.003747
7,2000,Placer County (California),API,All Ages Total,83.479892,84.728982,82.265971
8,2000,Placer County (California),Black,All Ages Total,77.717425,79.585958,76.054384
9,2000,Placer County (California),Latino,All Ages Total,80.226046,81.432906,79.128686


,Year,County,Race,Age,Estimate,Upper,Lower
0,2000,SACOG Total,AIAN,All Ages Total,77.402344,81.761611,73.940735
1,2000,SACOG Total,API,All Ages Total,81.497430,82.935446,80.148106
2,2000,SACOG Total,Black,All Ages Total,75.089799,77.067364,73.357794
3,2000,SACOG Total,Latino,All Ages Total,80.662771,82.189397,79.328314
4,2000,SACOG Total,Total,All Ages Total,77.149558,77.611004,76.692845
5,2000,SACOG Total,White,All Ages Total,76.694223,77.190381,76.198307
6,2001,SACOG Total,AIAN,All Ages Total,77.026635,81.069600,73.776366
7,2001,SACOG Total,API,All Ages Total,81.418666,82.747921,80.154009
8,2001,SACOG Total,Black,All Ages Total,75.035488,76.874402,73.421967
9,2001,SACOG Total,Latino,All Ages Total,80.434756,81.801115,79.215892


Should be noted, some of the calculations for estimates, upper, and lower are slightly different compared to Seth's values. For instance, I had an estimate of 80.431622 for El Dorado County (California) AIAN in 2000. Conversely, Seth had 80.1633875910602. Another example, for El Dorado API 2000, I had 83.191716 and Seth had 83.19933999842. 